In [4]:
import os
# Set this BEFORE importing transformers or langchain
# os.environ['HF_HOME'] = './models_cache'

In [ ]:
import subprocess
import sys
from dotenv import load_dotenv
import numpy as np
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
from urllib.request import urlretrieve
from langchain_openai import ChatOpenAI


The RetrievalQA import error: 

Option 1: Use langchain-classic (Recommended for existing code)

If you are maintaining legacy code that relies on RetrievalQA, you must import it from the langchain_classic package. This package preserves the v0.x API. 

In [7]:
from langchain_classic.chains import RetrievalQA
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

Option 2: Migrate to the New API (LangChain v1.0+)

The RetrievalQA class is deprecated in favor of a more modular approach using create_retrieval_chain and create_stuff_documents_chain. This is the recommended path for new projects or long-term maintenance. 

In [ ]:
# from langchain_classic.chains import create_retrieval_chain

#from langchain_classic.chains.combine_documents import create_stuff_docume

"""


def create_rag_chain(vector_db, llm):
    """Create RAG chain using the modern LCEL implementation"""
    
    # 1. Create the retriever
    retriever = vector_db.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}
    )

    # 2. Define the system prompt
    # Note: Modern chains prefer 'input' and 'context' as variable names
    system_prompt = (
        "Use the following pieces of context to answer the question at the end. "
        "Please follow the following rules:\n"
        "1. If you don't know the answer, don't try to make up an answer. "
        "Just say 'I can't find the final answer but you may want to check the following links'.\n"
        "2. If you find the answer, write the answer in a concise way with five sentences maximum.\n\n"
        "{context}"
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "{input}"),
        ]
    )

    # 3. Create the document chain (The "Stuff" part)
    question_answer_chain = create_stuff_documents_chain(llm, prompt)

    # 4. Create the final retrieval chain
    # This combines the retriever and the document chain
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)

    return rag_chain


"""

In [8]:
# Load the variables from .env into the system environment
load_dotenv()

True

In [9]:
# Load and split document

def load_and_split_single_pdf(pdf_path):
    """Load a single PDF and split it into chunks.""" 
    loader = PyMuPDFLoader(pdf_path)
    doc = loader.load()
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50)
    chunks = text_splitter.split_documents(doc)

    if chunks: 
        print(f"doc succesfully chunked")
    else:
        print("chunking failed")
    return chunks

chunks = load_and_split_single_pdf("cv.pdf")
print(chunks[1].page_content[:500])


doc succesfully chunked
1 
 
I. Résumé 
Titulaire d’un doctorat en informatique de l’Université Côte d’Azur (qualifié – 
section informatique-27), ainsi que d’un double Master en Sciences des 
Données et en Informatique de Gestion, je candidate avec détermination au 
poste de Maître de Conférences au sein de l'université d'Aix-Marseille. 
 
À travers ce CV analytique, je présente de manière détaillée mon profil, mon 
parcours ainsi que mon projet d’intégration au sein de votre établissement.


In [11]:
# chunk pdfs (for file path or directory)

"""" def load_and_split_documents(path):

    # Detect if it's a file or directory
    if os.path.isfile(path) and path.endswith(".pdf"):
        loader = PyMuPDFLoader(path)
    elif os.path.isdir(path):
        loader = PyPDFDirectoryLoader(path)
    else:
        raise ValueError("Invalid path: provide a PDF file or folder of PDFs.")

    docs_before_split = loader.load() """

'" def load_and_split_documents(path):\n\n    # Detect if it\'s a file or directory\n    if os.path.isfile(path) and path.endswith(".pdf"):\n        loader = PyMuPDFLoader(path)\n    elif os.path.isdir(path):\n        loader = PyPDFDirectoryLoader(path)\n    else:\n        raise ValueError("Invalid path: provide a PDF file or folder of PDFs.")\n\n    docs_before_split = loader.load() '

In [10]:
# initialize the embedding model and create FAISS vector store

def create_vector_store(chunks):
    embeddings = HuggingFaceBgeEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2",
                                          model_kwargs={'device':'cpu'},
                                          encode_kwargs={"normalize_embeddings":True})
    vector_store = FAISS.from_documents(chunks,embeddings)
    return vector_store

vector_store = create_vector_store(chunks)
print(f"vector store created with {vector_store.index.ntotal} vectors")


C:\Users\hady_\AppData\Local\Temp\ipykernel_28044\1040423773.py:4: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2",


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vector store created with 62 vectors


In [8]:
# save vector store 
vector_store.save_local("faiss_index")

In [11]:
# set openai key
llm = ChatOpenAI(model="gpt-4o-mini", max_tokens = 100)

In [12]:
# create rag chain
def create_rag_chain(vector_store, llm):
    retriever = vector_store.as_retriever(search_type = "similarity",
                                          search_kwargs={"k":3})
    
    prompt_template = """ 
    Use the following pieces of context to answer the question at the end. Please follow the following rules:
1. If you don't know the answer, don't try to make up an answer. Just say "I can't find the final answer but you may want to check the following links".
2. If you find the answer, write the answer in a concise way with five sentences maximum.

{context}

Question: {question}

Helpful Answer:"""
    
    prompt = PromptTemplate(template=prompt_template,
                            input_variables=["context","question"])
    
    rag_chain = RetrievalQA.from_chain_type(llm=llm,
                                            retriever=retriever,
                                            return_source_documents=True,
                                               chain_type_kwargs={"prompt": prompt})
    return rag_chain



In [13]:
rag = create_rag_chain(vector_store, llm)

In [13]:
response  = rag.invoke({"query": "what is the name of the person in the CV?"})
print(response["result"])

The name of the person in the CV is Hadi Nasser.


In [14]:
from ipywidgets import Text, Button, Output
import ipywidgets as widgets


In [ ]:

# Create input widget
question_input = widgets.Text(description='Question:')
ask_button = widgets.Button(description='Ask')
output = widgets.Output()

def on_button_click(b):
    with output:
        result = rag.invoke({"query": question_input.value})
        #result = ("wait until the rag model is setup")
        print(result["result"])

ask_button.on_click(on_button_click)
display(question_input, ask_button, output)

# from cmd: pip freeze > requirements.txt 